# 手写MNIST，理解底层逻辑
自定义训练全流程

## 一、先看全局地图

训练一个模型，本质上是这 4 步不断循环：

```
原始数据
   ↓  【Dataset】把数据组织成"可按索引取"的形式
单条样本
   ↓  【DataLoader】把单条样本打包成 batch
一个 batch
   ↓  【训练循环】前向传播 → 计算 loss → 反向传播 → 更新参数
模型参数更新
   ↓  【日志记录】把 loss/acc 写入 CSV
```

先有这张地图，再往下看每一层。

## 二、Dataset：告诉 PyTorch "第 i 条数据长什么样"

### 2.1 为什么需要自定义

PyTorch 不知道你的数据存在哪、是什么格式（图片/文本/音频）、怎么读取。
你需要写一个"适配器"，告诉 PyTorch：**给你一个索引 i，你返回第 i 条数据**。

这个适配器就是继承 `torch.utils.data.Dataset` 的类。

### 2.2 底层逻辑：必须实现的 3 个方法

| 方法             | 作用                          | 必须实现？ |
| ---------------- | ----------------------------- | ---------- |
| `__init__`       | 初始化，加载原始数据到内存    | 是         |
| `__len__`        | 告诉 PyTorch 数据集有多少条   | 是         |
| `__getitem__(i)` | 根据索引 i 返回第 i 条 (x, y) | 是         |

DataLoader 在内部会不断调用 `__getitem__`，按索引取数据，所以你只需要实现"取第 i 条"这一件事。

### 2.3 动手写 MNIST Dataset
**第一步：加载原始数据（借用 torchvision 下载，但自己管理数据）**

In [1]:
from torchvision import datasets

#   下载原始数据 MNIST
raw=datasets.MNIST(root='./data',train=True,download=True)

train_x=raw.data.float()/255.0      # shape：（6000,28,28），归一化到[0,1]
train_y=raw.targets         # shape：（6000，），  [0,9]

100.0%
100.0%
100.0%
100.0%


**第二步：自定义 Dataset 类**

In [2]:
import torch
from torch.utils.data import Dataset

class MNISTDataset(Dataset):
    def __init__(self,images,labels):
        self.images=images
        self.labels=labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self,idx):
        x=self.images[idx]
        x=x.unsqueeze(0)
        y=self.labels[idx]
        return x,y

**为什么要 `unsqueeze(0)`？**
`Conv2d` 要求输入是 `(batch, channel, H, W)` 4 维。MNIST 是灰度图，channel=1，
所以单张图从 `(28, 28)` 变成 `(1, 28, 28)`，DataLoader 会自动帮你堆成 `(batch, 1, 28, 28)`。

**第三步：实例化**

In [3]:
train_dataset=MNISTDataset(train_x,train_y)
print(len(train_dataset))
x,y=train_dataset[0]
print(x.shape,y)

60000
torch.Size([1, 28, 28]) tensor(5)


 ## 三、DataLoader：自动打包成 batch

### 3.1 DataLoader 做了什么

你的 Dataset 只能一条一条取数据。DataLoader 在这之上做了 3 件事：

1. **shuffle（打乱）**：每个 epoch 开始前打乱索引顺序，防止模型记住顺序
2. **batch（打包）**：把 `batch_size` 条数据合并成一个 batch，利用 GPU 并行计算
3. **num_workers（多线程预加载）**：训练 GPU 的同时，CPU 提前读下一个 batch，不让 GPU 等数据

### 3.2 关键参数

In [4]:
from torch.utils.data import DataLoader

train_loader=DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

### 3.3 迭代一个 batch

In [5]:
for batch_x,batch_y in train_loader:
    print(batch_x.shape)
    print(batch_y.shape)
    break

torch.Size([64, 1, 28, 28])
torch.Size([64])


每次循环，DataLoader 自动返回一个 batch 的 x 和 y。
**你不需要手动管理索引，这正是 DataLoader 的价值。**

## 四、定义模型

In [6]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        return self.net(x)

### 4.1 nn.Sequential

nn.Sequential 就像一个“管道”：你把一层一层按顺序塞进去，数据进去后自动按顺序流经每一层，你不需要在forward里手写

唯一的区别：

你之前这里是 两个独立的 Linear，中间还夹了一个 Dropout。

现在这个流水线版本没有 Dropout。如果想加上，只要在 nn.ReLU() 和 nn.Linear(128, 10) 之间插入一行 nn.Dropout(0.5), 即可。

### 两种写法的对比
零件式	流水线筒式（Sequential）
怎么定义层	每个层单独命名：self.conv1 = ...	全塞进 nn.Sequential(...)
forward 写法	手动一步步调用	一行 return self.net(x)
灵活性	高，可以自由跳跃、分支	低，只能顺序执行
简洁性	啰嗦	极简
适用场景	复杂网络（ResNet、Transformer等）	简单直筒网络（你现在写的这种）

### 4.2 用代码验证形状

In [7]:
model=SimpleCNN()

dummy=torch.zeros(1,1,28,28)

x=dummy
for layer in model.net:
    x=layer(x)
    print(f'{layer.__class__.__name__:20s} -> {tuple(x.shape)}')

Conv2d               -> (1, 32, 28, 28)
ReLU                 -> (1, 32, 28, 28)
MaxPool2d            -> (1, 32, 14, 14)
Conv2d               -> (1, 64, 14, 14)
ReLU                 -> (1, 64, 14, 14)
MaxPool2d            -> (1, 64, 7, 7)
Flatten              -> (1, 3136)
Linear               -> (1, 128)
ReLU                 -> (1, 128)
Linear               -> (1, 10)


**这段代码是调试利器**，搭任何新模型都可以用它验证形状。

## 五、初始化

In [8]:
device=torch.device('cuda'if torch.cuda.is_available() else 'cpu')
model=SimpleCNN().to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
criterion=nn.CrossEntropyLoss()

## 六、训练循环：一次迭代发生了什么

### 6.1 梯度下降的完整流程（概念层）

```
前向传播：x → model → output（预测值）
计算 loss：output 和真实 y 差多少
反向传播：自动计算每个参数对 loss 的梯度
参数更新：每个参数 -= 学习率 × 梯度
```

PyTorch 的"自动微分"帮你完成反向传播，你只需要写前向传播和调用 `.backward()`。

### 6.2 逐行拆解代码

In [9]:
import csv
import os


os.makedirs('../logs', exist_ok=True)
num_epochs=3

with open('../logs/train_log.csv', 'w', newline='') as f:
    writer=csv.writer(f)
    writer.writerow(['epoch','train_loss','train_acc'])

    for epoch in range(num_epochs):
        model.train()
        total_loss=0
        correct=0
        total=0

        for batch_x,batch_y in train_loader:
            batch_x,batch_y=batch_x.to(device),batch_y.to(device)
            optimizer.zero_grad()
            output=model(batch_x)
            loss=criterion(output,batch_y)
            loss.backward()
            optimizer.step()

            total_loss+=loss.item()
            prediction=output.argmax(dim=1)
            correct+=(prediction==batch_y).sum().item()
            total+=len(batch_y)

        ave_loss=total_loss/len(train_loader)
        accuracy=correct/total
        writer.writerow([epoch+1,f'{ave_loss:.4f}',f'{accuracy:.4f}'])
        print(f'Epoch{epoch+1}/{num_epochs} | Loss:{ave_loss:.4f} | Accuracy:{accuracy:.4f}')

print('训练完成，日志已保存到 logs/train_log.csv')

Epoch1/3 | Loss:0.1667 | Accuracy:0.9494
Epoch2/3 | Loss:0.0504 | Accuracy:0.9841
Epoch3/3 | Loss:0.0356 | Accuracy:0.9891
训练完成，日志已保存到 logs/train_log.csv


### 6.3 为什么 `zero_grad()` 必须在最前面

PyTorch **默认会累加梯度**，不会自动清零。
如果不清零，第 2 个 batch 的梯度会叠加到第 1 个 batch 的梯度上，参数更新就错了。
所以每次处理新的 batch 前，必须先清零。

### 6.4 `model.train()` 和 `model.eval()` 的区别

|           | `model.train()`          | `model.eval()`          |
| --------- | ------------------------ | ----------------------- |
| 使用时机  | 训练时                   | 验证/测试时             |
| BatchNorm | 用当前 batch 的均值/方差 | 用训练时统计的均值/方差 |
| Dropout   | 随机丢弃神经元           | 关闭，全部保留          |

```python
model.train()   # 开启训练模式，写在 epoch 循环外
for batch_x, batch_y in train_loader:
    ...         # 训练

model.eval()    # 切换评估模式
with torch.no_grad():   # 关闭梯度计算，节省内存
    for batch_x, batch_y in val_loader:
        ...     # 验证
```

---

## 七、常见报错速查

| 报错                                                   | 原因                                  | 解决                                  |
| ------------------------------------------------------ | ------------------------------------- | ------------------------------------- |
| `RuntimeError: Expected 4D tensor`                     | 忘了 `unsqueeze(0)` 加 channel 维度   | 在 `__getitem__` 里加 `.unsqueeze(0)` |
| `Expected input batch_size to match target batch_size` | x 和 y 的 batch 大小不一致            | 检查 Dataset 的返回值                 |
| `RuntimeError: ...on CPU...CUDA`                       | 数据在 CPU，模型在 GPU                | 在训练循环里加 `.to(device)`          |
| `BrokenPipeError`（Windows）                           | `num_workers > 0` 在 Windows 上有问题 | 设 `num_workers=0`                    |
| loss 不下降                                            | 可能没调用 `optimizer.zero_grad()`    | 检查训练循环顺序                      |